# Notbook 004 — ¿Cuáles operaciones necesita una neurona?

**Hipótesis:** seleccionar únicamente las operaciones cuadráticas necesarias puede reducir parámetros sin perder precisión apreciable en tareas específicas.

- Diez semillas y cinco tareas sintéticas.
- Entrenamiento: 768 ejemplos; validación: 384; prueba: 2048.
- Evaluamos los 32 subconjuntos de cinco términos: x1, x2, x1², x1·x2 y x2².
- Elegimos el subconjunto más pequeño dentro de 1 punto porcentual de precisión y 0,02 de pérdida logarítmica respecto del modelo completo en **validación**.
- Incluimos modelo lineal y control aleatorio del mismo tamaño.

**Importante:** reducir parámetros no prueba equivalencia exacta; buscar 32 modelos puede costar más entrenamiento que usar el modelo completo. Los resultados no se extrapolan a cualquier red.

In [ ]:
import warnings, time, itertools
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_moons
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import log_loss

warnings.filterwarnings('ignore', category=ConvergenceWarning)
OUTPUT_DIR = Path('/content') if Path('/content').exists() else Path('/mnt/data')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SEEDS = [11,23,37,41,53,67,71,83,97,101]
TASKS = ['XOR','Circulos','Lineal','Anillo','Lunas']
TERMS = ('x1','x2','x1^2','x1*x2','x2^2')
SETS = [s for k in range(6) for s in itertools.combinations(range(5),k)]
SELECTION_ACCURACY_TOLERANCE = 0.01
SELECTION_NLL_TOLERANCE = 0.02

def crear_datos(task,n,seed):
    if task == 'Lunas':
        x,y = make_moons(n_samples=n,noise=0.10,random_state=seed)
        return (x-np.array([0.5,0.25]))/1.5,y
    rng = np.random.default_rng(seed)
    x = rng.uniform(-1,1,(n,2))
    a,b = x.T
    if task == 'XOR': y = a*b > 0
    elif task == 'Circulos': y = a*a+b*b < .75**2
    elif task == 'Lineal': y = a+.6*b > 0
    elif task == 'Anillo': y = ((a*a+b*b>.35**2)&(a*a+b*b<.8**2))
    else: raise ValueError(task)
    return x,np.asarray(y,dtype=np.int64)

def features(x):
    a,b=x.T
    return np.column_stack((a,b,a*a,a*b,b*b))

def train_subset(X,y,subset):
    if not subset: return (None,None,float(y.mean()+1e-7))
    scaler=StandardScaler().fit(X[:,subset])
    clf=LogisticRegression(C=100.0,max_iter=500,solver='lbfgs',tol=1e-7)
    clf.fit(scaler.transform(X[:,subset]),y)
    return clf,scaler,None

def predict(model,X,subset):
    clf,scaler,const=model
    if const is not None:
        return np.full(X.shape[0],np.clip(const,1e-7,1-1e-7))
    return clf.predict_proba(scaler.transform(X[:,subset]))[:,1]

def metric(p,y):
    return float(((p>=.5)==y).mean()),float(log_loss(y,p,labels=[0,1]))

def run():
    rows=[]; selections=[]
    start=time.perf_counter()
    for task in TASKS:
        for seed in SEEDS:
            xtrain,ytrain=crear_datos(task,768,1000+seed)
            xval,yval=crear_datos(task,384,3000+seed)
            xtest,ytest=crear_datos(task,2048,2000+seed)
            Ftrain,Fval,Ftest=map(features,(xtrain,xval,xtest))
            fitted={}
            for subset in SETS:
                m=train_subset(Ftrain,ytrain,subset)
                acc,nll=metric(predict(m,Fval,subset),yval)
                fitted[subset]=(m,acc,nll)
            full=(0,1,2,3,4)
            val_full_acc,val_full_nll=fitted[full][1:]
            eligible=[s for s in SETS
                      if fitted[s][1]>=val_full_acc-SELECTION_ACCURACY_TOLERANCE
                      and fitted[s][2]<=val_full_nll+SELECTION_NLL_TOLERANCE]
            selected=min(eligible,key=lambda s:(len(s),fitted[s][2],s))
            same_size=[s for s in SETS if len(s)==len(selected)]
            control=same_size[np.random.default_rng(70000+seed).integers(len(same_size))]
            models={'Lineal':(0,1),'Cuadratica-6':full,'Seleccionada':selected,
                    'Control-aleatorio':control}
            for name,s in models.items():
                model,valacc,valnll=fitted[s]
                acc,nll=metric(predict(model,Ftest,s),ytest)
                rows.append(dict(tarea=task,semilla=seed,modelo=name,
                                terminos=len(s),parametros=1+len(s),
                                precision_test=acc,perdida_test=nll,
                                precision_validacion=valacc,perdida_validacion=valnll,
                                operaciones='|'.join(TERMS[i] for i in s)))
            selections.append(dict(tarea=task,semilla=seed,
                                   seleccion='|'.join(TERMS[i] for i in selected),
                                   n_terminos=len(selected),
                                   val_full_accuracy=val_full_acc,
                                   val_full_nll=val_full_nll))
        print('Completada',task,'segundos',round(time.perf_counter()-start,1))
    data=pd.DataFrame(rows); sel=pd.DataFrame(selections)
    summary=data.groupby(['tarea','modelo'],sort=False).agg(
        precision_media=('precision_test','mean'),
        desviacion=('precision_test','std'),
        perdida_media=('perdida_test','mean'),
        parametros_medios=('parametros','mean'),
        min_parametros=('parametros','min'),
        max_parametros=('parametros','max')).reset_index()
    data.to_csv(OUTPUT_DIR/'Notbook_004_resultados.csv',index=False)
    sel.to_csv(OUTPUT_DIR/'Notbook_004_selecciones.csv',index=False)
    summary.to_csv(OUTPUT_DIR/'Notbook_004_resumen.csv',index=False)
    print(summary.to_string(index=False))
    return data,sel,summary


In [ ]:
resultados, selecciones, resumen = run()
display(resumen.round(4))

In [ ]:
print('Operaciones elegidas por tarea:')
for tarea,g in selecciones.groupby('tarea',sort=False):
    print(tarea, g.seleccion.value_counts().to_dict())

In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(10,4))
for model,g in resumen.groupby('modelo'):
    ax.plot(g.tarea,100*g.precision_media,marker='o',label=model)
ax.set(ylabel='Precisión prueba (%)',title='Diez semillas, cinco tareas')
ax.grid(alpha=.2); ax.legend(ncol=2); plt.show()

## Conclusión

La selección de términos puede disminuir la complejidad inferencial de esta familia cuadrática. XOR necesita esencialmente la interacción x1·x2 y círculos depende de términos radiales; anillos y lunas siguen evidenciando límites de capacidad. **No** hay una neurona universalmente superior ni equivalencia exacta general. Propuesta para Notbook 005: explorar la mínima operación adicional que permite representar dos fronteras radiales.